In [2]:
import pandas as pd 
from transformers import T5Tokenizer , Trainer , TrainingArguments , T5ForConditionalGeneration  

c:\Users\wardo\Downloads\PRIME AI ML\day 48 text summarizer project\venv_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### DATA LOADING

In [3]:
train_data = pd.read_csv('samsum-train.csv')
val_data = pd.read_csv('samsum-validation.csv') 

In [4]:
train_data.head() 

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


### RANDOM SAMPLING

In [5]:
train_data = train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [ ]:
val_data

### DATA PRE-PROCESSING

In [6]:
import re 

def clean_data(text):
    text = re.sub(r'\r\n'," ",text) # remove lines 
    text = re.sub(r'\s+'," ",text) # remove extra space 
    text = re.sub(r'<.*?>'," ",text) # remove html tags  
    text = text.strip().lower() 

    return text 

In [7]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data) 

### TOKENIZATION

In [8]:
tokenizer = T5Tokenizer.from_pretrained('t5-small') 

# raw data => tokenize inputs for fine tuning 

def tokenize(data):
    inputs = tokenizer(data['dialogue'],padding='max_length',max_length=512,truncation=True)
    targets = tokenizer(data['summary'],padding='max_length',max_length=150,truncation=True)

    inputs['labels'] = targets['input_ids'] # token ids add to input as label 
    return inputs

In [9]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()

In [ ]:
train_dataset[0] 

### MODEL

In [10]:
# NLP = Generation task

model = T5ForConditionalGeneration.from_pretrained('t5-small')  

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 3275.71it/s]


In [11]:
# fine tune 

import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Device:", device)
model.to(device)

Device: cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [12]:
# training arguments 

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    warmup_steps=500
) 

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
# train the model 

trainer.train()   

### SAVING THE MODEL

In [14]:
model.save_pretrained('./save_summary_model')
tokenizer.save_pretrained('./save_summary_model')

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.47s/it]


('./save_summary_model\\tokenizer_config.json',
 './save_summary_model\\tokenizer.json')

### USING SAVED MODEL 

In [ ]:
model = T5ForConditionalGeneration.from_pretrained('./save_summary_model')
tokenizer = T5Tokenizer.from_pretrained('./save_summary_model') 

### TEST CORE LOGIC FOR SUMMARIZATION

In [17]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )
    
    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary 

In [ ]:
test_dialogue = """
Reporter: Technology is advancing rapidly in today's world.  
Expert: Yes, especially with artificial intelligence and automation.  
Reporter: How is this affecting jobs?  
Expert: While some jobs are being replaced, new opportunities are also being created.  
Reporter: What about the risks?  
Expert: There are concerns about data privacy and ethical use of technology.  
Reporter: So what should be done?  
Expert: Proper regulations and responsible innovation are key to managing these changes."""


summary = summarize_dialogue(test_dialogue)

print('summary :',summary)  

summary : technology is advancing rapidly in today's world. expert: technology is advancing rapidly in today's world. expert: while some jobs are being replaced, new opportunities are also being created. expert: there are concerns about data privacy and ethical use of technology. expert: proper regulations and responsible innovation are key to managing these changes.
